# 05 Backpropagation Detailed

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- See how gradients flow backward through the network (backpropagation)
- Use TensorFlow's GradientTape to compute gradients for a small model
- Understand why we use backprop instead of guessing weights

---

## 🌍 Real life

**Where is this used?** Every time you train a neural network (digit recognition, image classification, NLP), the **optimizer** uses **gradients** from backpropagation to update the weights.

**In this notebook we use** **backpropagation** (via GradientTape) to **compute gradients of the loss with respect to the weights**. We use it **instead of** guessing or randomly changing weights **because** gradients tell us **in which direction** to change each weight to reduce the loss.

**📌 Covers slide(s):** **23** — Keras, training flow (how gradients and optimizer work). *Do this notebook after that slide.*

---

**Before starting:** Run the imports cell below. Requires TensorFlow.


## Theory (short)

- **Forward pass:** input → layers → output → loss.
- **Backpropagation:** we compute how much each weight contributed to the loss (the **gradient** of loss w.r.t. that weight) using the **chain rule**.
- **Gradients flow backward:** from output/loss back through each layer to the weights.
- **Optimizer** (e.g. Adam) uses these gradients to update weights: `new_weight = old_weight - learning_rate * gradient`.
- **Data flow:** forward: input → layers → loss. Backward: loss → gradients → update weights.

### Key math (required)

- **Chain rule:** If \(L\) is the loss and \(w\) is a weight, \(\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a}\cdot\frac{\partial a}{\partial z}\cdot\frac{\partial z}{\partial w}\) where \(a\) is the layer output and \(z\) is the pre-activation. Backprop computes these derivatives layer by layer from output to input.
- **Gradient update:** \(w_{\text{new}} = w_{\text{old}} - \eta\,\frac{\partial L}{\partial w}\) with learning rate \(\eta\).
- For full derivation (e.g. matrix form, multi-layer), see the lecture slides or Goodfellow et al., *Deep Learning*, Ch. 6.

**The steps below put this theory into code.**

💡 **If this is unclear:** Run the next few cells and look at the printed **loss** and **gradient shapes**; the code is doing the chain rule. The "loss before vs after one update" cell shows that the optimizer moved the weights in the right direction. If still stuck, tell your instructor: "I didn't get the backprop / gradient part in notebook 05" so they can go over that cell with you.


## 📥 Inputs & 📤 Outputs

**Inputs:** TensorFlow, NumPy, MNIST (we load it and take a batch of 32 images). We build a one-layer model in the notebook.

**Dataset:** Real — MNIST (small batch for backprop demo).

**Outputs:** Loss value (one number), gradient shapes and a sample of gradient values, then "loss before update" and "loss after one update" (loss should go down), and a bar chart comparing loss before vs after one gradient update.


In [ ]:
# Step 1: Imports — PyTorch autograd for backpropagation
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
print(f'PyTorch {torch.__version__}')
print('✅ Imports OK.')

In [ ]:
# Step 2: Create a tiny synthetic batch (same shape as MNIST: 5 × 784)
# Using synthetic data means no download and the cell runs instantly.
torch.manual_seed(0)
x_flat  = torch.randn(5, 784)         # 5 'images', 784 pixels each
y_batch = torch.randint(0, 10, (5,))  # random digit labels 0–9
print('Input batch shape:', x_flat.shape)
print('Labels:', y_batch.tolist())

In [ ]:
# Step 3: Build a tiny model (one hidden layer) so gradients are easy to inspect
model = nn.Sequential(
    nn.Linear(784, 16),
    nn.ReLU(),
    nn.Linear(16, 10),
)
print(model)

In [ ]:
# Step 4: Forward pass + compute loss; PyTorch builds the computational graph automatically.
# Calling .backward() computes all gradients (equivalent to GradientTape in TF).
criterion = nn.CrossEntropyLoss()
logits = model(x_flat)              # forward pass
loss   = criterion(logits, y_batch)  # compute loss
loss.backward()                      # backpropagation — fills .grad for all parameters
print(f'Loss before update: {loss.item():.4f}')
print('Gradients computed for', sum(1 for p in model.parameters() if p.grad is not None), 'parameter tensors')

In [ ]:
# (Next: Step 5 — look at actual gradient values.)

In [ ]:
# Step 5: Inspect a sample of gradient values from the first linear layer
grad_sample = model[0].weight.grad[0, :10]  # first 10 weights of first neuron
print('Gradient sample (first 10 values of first neuron weight):')
print(grad_sample.numpy().round(5))
# Non-zero values confirm that backprop is working — the network 'knows' how to improve.

In [ ]:
# Step 6: One manual update step — loss should decrease
optimizer = optim.Adam(model.parameters(), lr=1e-3)
optimizer.step()     # apply the gradients computed in Step 4
optimizer.zero_grad()

# Recompute loss after update to confirm it went down
with torch.no_grad():
    loss_after = criterion(model(x_flat), y_batch)
print(f'Loss before update: {loss.item():.4f}')
print(f'Loss after  update: {loss_after.item():.4f}')
print('Loss decreased:', loss_after.item() < loss.item())

## 🌍 Real-World Worked Example — House Price Prediction (Manual Backprop)

**Industry context:** Real estate platforms (Zillow, Property Finder) use regression models  
trained with gradient descent to estimate property prices.

Below we implement a 1-hidden-layer network **with manual backpropagation** to predict house prices.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

np.random.seed(42)
# ── Synthetic house data (size in m², price in 100k SAR) ──────────────────
n = 200
X = np.random.uniform(50, 300, (n, 1)).astype(np.float32)       # size
y = (2.5 * X + np.random.randn(n,1)*30).astype(np.float32)      # price

# Normalise
X_n = (X - X.mean()) / X.std()
y_n = (y - y.mean()) / y.std()

# ── Manual 1-layer network ─────────────────────────────────────────────────
def relu(z): return np.maximum(0, z)
def relu_d(z): return (z > 0).astype(float)

W1 = np.random.randn(1, 16) * 0.1;  b1 = np.zeros((1, 16))
W2 = np.random.randn(16, 1) * 0.1;  b2 = np.zeros((1, 1))
lr = 0.01;  losses = []

for _ in range(500):
    # Forward
    z1 = X_n @ W1 + b1          # (n, 16)
    a1 = relu(z1)
    z2 = a1 @ W2 + b2            # (n, 1)
    loss = ((z2 - y_n)**2).mean()
    losses.append(loss)

    # ── Backpropagation (chain rule, step by step) ─────────────────────────
    dL_dz2 = 2*(z2 - y_n) / n
    dL_dW2 = a1.T @ dL_dz2
    dL_db2 = dL_dz2.sum(0, keepdims=True)
    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * relu_d(z1)
    dL_dW1 = X_n.T @ dL_dz1
    dL_db1 = dL_dz1.sum(0, keepdims=True)

    # Gradient descent step
    W2 -= lr * dL_dW2;  b2 -= lr * dL_db2
    W1 -= lr * dL_dW1;  b1 -= lr * dL_db1

print(f"Final MSE loss: {losses[-1]:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(losses); plt.title("Loss Curve — House Price Model"); plt.xlabel("Epoch")
plt.subplot(1, 2, 2)
pred_y = (relu(X_n @ W1 + b1) @ W2 + b2) * y.std() + y.mean()
plt.scatter(X, y, alpha=0.3, label="Real data")
idx = X[:,0].argsort()
plt.plot(X[idx], pred_y[idx], 'r-', lw=2, label="Model prediction")
plt.xlabel("House size (m²)"); plt.ylabel("Price (100k SAR)"); plt.legend()
plt.title("Real-World: House Price Prediction")
plt.tight_layout(); plt.show()

---

## 🧩 Mini-exercise

**Try it:** In a new code cell, print the gradient of the first layer's kernel (e.g. `gradients[0]`) and its shape. Then apply the optimizer step again (same cell or new) and print the loss again—does it decrease further?

---

## ✅ Summary

**What you did:**
- Loaded a small batch of MNIST and built a tiny one-layer model.
- Used **GradientTape** to run a forward pass and compute **gradients** (backpropagation).
- Checked gradient shapes and one manual update step so loss decreased.

**In real life you'd also:** train for many steps (model.fit does forward + backprop + update in a loop); here we did one step to see the mechanism.

**The main idea:** Backpropagation gives us gradients of the loss w.r.t. each weight; the optimizer uses them to update weights so the loss goes down.

**Next:** `06_optimization_techniques` compares different optimizers (SGD, Adam, etc.) and how they use these gradients.


## 📚 References & Further Reading

**Papers:**
- Rumelhart, Hinton & Williams (1986) — [Learning representations by back-propagating errors](https://www.nature.com/articles/323533a0)

**Tutorials:**
- [Andrej Karpathy — micrograd](https://github.com/karpathy/micrograd) (build backprop from scratch in 150 lines)
- [PyTorch Autograd Tutorial](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)

**State-of-the-Art:** Backpropagation is the engine behind GPT-4, Stable Diffusion, and every modern neural network.